# Pair seed explorer

Pick any pair and any seed parameters (the `daniel_filter` seed format), and this notebook renders the fifty-pairs style 4-panel diagnostic plus the single-pair eval performance (train 251-500 / test 501-750, eval-style accounting with 1bp fees).

- Simulation functions come from `tune_daniel_pairs.py` and replicate `daniel_filter.PairAgent` exactly (lagged-beta window z, ADF/kappa gates, hysteresis with flip).
- Edit **cell 3** and re-run cells 3-5. Set `USE_SEED = True` to prefill from the pair's live seed in `daniel_filter.py`, then override any field.


In [7]:
import sys
if "research" not in sys.path:
    sys.path.append("research")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import daniel_filter as dfil
from daniel_filter import FiltCfg, TradeCfg, I as TICKER
from tune_daniel_pairs import (
    TRAIN_START, TRAIN_END, TEST_START, TEST_END,
    TRAIN_DECIDE_START, TEST_DECIDE_START,
    record_paths, gate_and_z_paths, trade_sim, pnl_path, window_stats,
)

prices = pd.read_csv("prices.txt", sep=r"\s+", header=0).to_numpy(float).T
INV = {v: k for k, v in TICKER.items()}

print("seeded pairs in daniel_filter (EV, z entry/exit/stop, q, h, k):")
for s in dfil.PAIR_CFGS:
    print(
        f"  {s.name:>10}  EV {s.priority:6.1f}  "
        f"z {s.trade.z_entry:.2f}/{s.trade.z_exit:.2f}/{s.trade.z_stop:.2f}  "
        f"q {s.filt.q_beta:.2e}  h {s.filt.h:4.1f}  k {s.filt.cusum_k:.2f}  "
        f"ab {s.filt.adapt_bars:2d}  snap {s.filt.snap_overshoot:.2f}"
    )


ModuleNotFoundError: No module named 'pair_seg_filter'

In [ ]:
# Refresh live seeds and simulator functions even in a long-running kernel.
import importlib
dfil = importlib.reload(dfil)
from . import tune_daniel_pairs as _pair_sim
_pair_sim = importlib.reload(_pair_sim)
record_paths = _pair_sim.record_paths
gate_and_z_paths = _pair_sim.gate_and_z_paths
trade_sim = _pair_sim.trade_sim
pnl_path = _pair_sim.pnl_path
window_stats = _pair_sim.window_stats

# ----- EDIT ME ---------------------------------------------------------
Y_TICKER = "HUXZ"       # y leg
X_TICKER = "ACAC"       # x leg
USE_SEED = True          # prefill from daniel_filter seeds when the pair is seeded

# Override any seed field here; leave empty to use the seed / defaults.
FILT_OVERRIDES = dict(
     #q_beta=70e-07, h=7.0, cusum_k=0.60, adapt_bars=6, snap_overshoot=1.40,
)
TRADE_OVERRIDES = dict(
    # z_entry=0.75, z_exit=0.02, z_stop=4.00, leg_scale=1.00,
    # require_adf=True, require_kappa=True,
)
# -----------------------------------------------------------------------

y_i, x_i = TICKER[Y_TICKER], TICKER[X_TICKER]
seed = None
if USE_SEED:
    seed = next((s for s in dfil.PAIR_CFGS if {s.y, s.x} == {y_i, x_i}), None)
if seed is not None and (seed.y, seed.x) != (y_i, x_i):
    y_i, x_i = seed.y, seed.x  # keep the seed's orientation
    Y_TICKER, X_TICKER = INV[y_i], INV[x_i]

filt_cfg = (seed.filt if seed else FiltCfg())._replace(**FILT_OVERRIDES)
trade_cfg = (seed.trade if seed else TradeCfg())._replace(**TRADE_OVERRIDES)
print(f"pair {Y_TICKER}/{X_TICKER}   seeded: {seed is not None}")
print("filt :", filt_cfg)
print("trade:", trade_cfg)


NameError: name 'dfil' is not defined

In [ ]:
ly, lx = np.log(prices[y_i]), np.log(prices[x_i])
py, px = prices[y_i], prices[x_i]

paths = record_paths(ly, lx, filt_cfg)
gates = gate_and_z_paths(ly, lx, paths["beta"])

# Train and test are replayed with independent hysteresis state, matching
# two separate eval.py runs; the train-state run also supplies the
# continuous cumulative curve for the plot.
sy_tr, sx_tr = trade_sim(gates, py, px, trade_cfg, TRAIN_DECIDE_START)
sy_te, sx_te = trade_sim(gates, py, px, trade_cfg, TEST_DECIDE_START)
train = window_stats(pnl_path(sy_tr, sx_tr, py, px), TRAIN_START, TRAIN_END)
test = window_stats(pnl_path(sy_te, sx_te, py, px), TEST_START, TEST_END)

full_pnl = pnl_path(sy_tr, sx_tr, py, px)
side = np.sign(sy_tr).astype(int)
side_te = np.sign(sy_te).astype(int)
blocked = np.zeros(len(py), dtype=bool)
if trade_cfg.require_adf:
    blocked |= gates["adf_block"]
if trade_cfg.require_kappa:
    blocked |= gates["kappa_block"]

def show(name, w):
    sharpe = (np.sqrt(250) * w["mean"] / w["std"]) if w["std"] > 0 else 0.0
    print(
        f"{name}: score {w['score']:8.2f}   mean ${w['mean']:7.2f}/day   "
        f"std {w['std']:7.2f}   Sharpe {sharpe:5.2f}   net ${w['net']:9.0f}"
    )

def transition_counts(position_side, start, end):
    scope = position_side[start - 2:end - 1]
    previous = position_side[start - 3:end - 2]
    entries = int(np.sum((scope != 0) & (previous == 0)))
    exits = int(np.sum((scope == 0) & (previous != 0)))
    flips = int(np.sum((scope != 0) & (previous != 0) & (scope != previous)))
    return entries, exits, flips

train_transitions = transition_counts(side, TRAIN_START, TRAIN_END)
test_transitions = transition_counts(side_te, TEST_START, TEST_END)
show("train 251-500", train)
show("test  501-750", test)
print(f"train entries/exits/flips {train_transitions}; "
      f"test {test_transitions}")
print(
    f"cuts {int(paths['cut'].sum())}   "
    f"days in market {int((side != 0).sum())}   "
    f"gate-blocked days {int(blocked[249:].sum())}"
)


In [ ]:
T = prices.shape[1]
days = np.arange(1, T + 1)
z = gates["z"]
cuts = np.flatnonzero(paths["cut"])
entries = np.flatnonzero((side != 0) & (np.roll(side, 1) == 0))
exits = np.flatnonzero((side == 0) & (np.roll(side, 1) != 0))
exits = exits[exits > 0]

C_Y, C_X = "#0072B2", "#E69F00"
C_CUT, C_LONG, C_SHORT = "#D55E00", "#009E73", "#CC79A7"
fig, axes = plt.subplots(
    4, 1, figsize=(12, 12), sharex=True,
    gridspec_kw={"height_ratios": [1.0, 1.0, 1.2, 1.2]},
)

ax = axes[0]
ax.plot(days, py / py[0], color=C_Y, lw=1.2, label=Y_TICKER)
ax.plot(days, px / px[0], color=C_X, lw=1.2, label=X_TICKER)
for d in cuts:
    ax.axvline(d + 1, color=C_CUT, ls="--", lw=0.9, alpha=0.8)
ax.set_ylabel("price / day 1")
ax.set_title(f"{Y_TICKER}/{X_TICKER} — seed explorer")
ax.legend(fontsize=8, loc="upper left", frameon=False)
ax.grid(True, alpha=0.25)

ax = axes[1]
w = 100
roll = np.full(T, np.nan)
for i in range(w, T):
    xs, ys = lx[i - w : i], ly[i - w : i]
    d = float(xs @ xs)
    if d > 1e-12:
        roll[i] = float(xs @ ys) / d
ax.plot(days, roll, color="#444444", lw=1.6, alpha=0.7, label="rolling 100d OLS (log)")
ax.plot(days, paths["beta"], color=C_Y, lw=1.3, label="filter beta")
for k, d in enumerate(cuts):
    ax.axvline(d + 1, color=C_CUT, ls="--", lw=0.9, alpha=0.8,
               label="CUSUM cut" if k == 0 else None)
ax.set_ylabel("hedge beta")
ax.legend(fontsize=8, loc="upper left", frameon=False)
ax.grid(True, alpha=0.25)

ax = axes[2]
zc = np.clip(z, -5, 5)
ax.fill_between(days, -5, 5, where=side > 0, color=C_LONG, alpha=0.12,
                step="pre", label="long spread")
ax.fill_between(days, -5, 5, where=side < 0, color=C_SHORT, alpha=0.14,
                step="pre", label="short spread")
ax.fill_between(days, -5, 5, where=blocked, color="#999999", alpha=0.10,
                step="pre", label="gate blocked")
ax.plot(days, zc, color="#444444", lw=0.9, label="window z (145d)")
for lvl in (trade_cfg.z_entry, -trade_cfg.z_entry):
    ax.axhline(lvl, color="#999999", lw=0.7, ls=":")
for lvl in (trade_cfg.z_stop, -trade_cfg.z_stop):
    ax.axhline(lvl, color=C_CUT, lw=0.7, ls=":")
ax.scatter(entries + 1, zc[entries], marker="^", s=28, color=C_LONG,
           zorder=3, label="entry")
ax.scatter(exits + 1, zc[exits], marker="x", s=28, color="#444444",
           zorder=3, label="exit")
ax.set_ylabel("z / position")
ax.set_ylim(-5.2, 5.2)
ax.legend(fontsize=7, loc="upper left", ncol=3, frameon=False)
ax.grid(True, alpha=0.25)

ax = axes[3]
pnl_days = np.arange(TRAIN_START, TEST_END + 1)
ax.plot(pnl_days, np.cumsum(full_pnl[TRAIN_START : TEST_END + 1]),
        color=C_Y, lw=1.5, label="cumulative PnL (continuous state)")
ax.axvline(TRAIN_END + 0.5, color="#444444", lw=1.0, ls=":", label="train | test")
ax.axhline(0, color="#999999", lw=0.7)
ax.annotate(f"train score {train['score']:.1f}", xy=(0.02, 0.90),
            xycoords="axes fraction", fontsize=9)
ax.annotate(f"test score {test['score']:.1f}", xy=(0.72, 0.90),
            xycoords="axes fraction", fontsize=9)
ax.set_ylabel("cumulative $")
ax.set_xlabel("day")
ax.legend(fontsize=8, loc="lower right", frameon=False)
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()


## Diagnostics

Run after cell 4 (they reuse `gates`, `sy_tr`, `sy_te`, `trade_cfg`).

In [ ]:
# ---- window participation: WHY does a window trade more or less? ----
# Splits each window into: days in market / blocked by a gate the pair
# honours / flat because |z| never reached z_entry.
import numpy as np

def _participation(gates, sy, start, end, trade):
    lo, hi = start - 1, end
    idx = slice(lo, hi)
    in_mkt = np.abs(sy[idx]) > 0
    adf = gates["adf_block"][idx] & bool(trade.require_adf)
    kap = gates["kappa_block"][idx] & bool(trade.require_kappa)
    flat = (~in_mkt) & (~adf) & (~kap)
    return {
        "in market": int(in_mkt.sum()),
        "gate: adf": int(adf.sum()),
        "gate: kappa": int(kap.sum()),
        "flat |z|<entry": int(flat.sum()),
        "mean |z|": float(np.abs(gates["z"][idx]).mean()),
    }

_tr = _participation(gates, sy_tr, TRAIN_START, TRAIN_END, trade_cfg)
_te = _participation(gates, sy_te, TEST_START, TEST_END, trade_cfg)
print(f"{Y_TICKER}/{X_TICKER}   (250 days per window)")
print()
print(f"{'':16s} {'train':>10} {'test':>10}")
for k in _tr:
    fmt = "{:>10.2f}" if "mean" in k else "{:>10d}"
    print(f"{k:16s} " + fmt.format(_tr[k]) + " " + fmt.format(_te[k]))
print()
print(f"{'score':16s} {train['score']:>10.2f} {test['score']:>10.2f}")
print(f"{'mean $/day':16s} {train['mean']:>10.2f} {test['mean']:>10.2f}")


In [ ]:
# ---- gate toggle: is a gate costing this pair good trades? ----
# The ADF prior-gate blocks whole stretches of a window on some pairs.
# This shows what the same seed earns with each gate switched off.
def _run(tc):
    a, b = trade_sim(gates, py, px, tc, TRAIN_DECIDE_START)
    c, d = trade_sim(gates, py, px, tc, TEST_DECIDE_START)
    tr = window_stats(pnl_path(a, b, py, px), TRAIN_START, TRAIN_END)
    te = window_stats(pnl_path(c, d, py, px), TEST_START, TEST_END)
    return (tr["score"], te["score"],
            int((np.abs(a[TRAIN_START-1:TRAIN_END]) > 0).sum()),
            int((np.abs(c[TEST_START-1:TEST_END]) > 0).sum()))

_variants = {
    "as seeded": trade_cfg,
    "no ADF gate": trade_cfg._replace(require_adf=False),
    "no kappa gate": trade_cfg._replace(require_kappa=False),
    "no gates": trade_cfg._replace(require_adf=False, require_kappa=False),
}
print(f"{'variant':>14} | {'train':>7} {'test':>7} | {'d.train':>7} {'d.test':>7}")
print("-" * 52)
_b = None
for _name, _tc in _variants.items():
    _r = _run(_tc)
    if _b is None:
        _b = _r
    _flag = " BOTH BETTER" if (_r[0] > _b[0] and _r[1] > _b[1]) else ""
    print(f"{_name:>14} | {_r[0]:7.2f} {_r[1]:7.2f} | {_r[2]:7d} {_r[3]:7d}{_flag}")


In [ ]:
# ---- z_entry sweep: does trading MORE in each window help? ----
# Lower z_entry = enter sooner = more days in market.  Shows both windows
# so a change that only helps one is immediately visible.
print(f"{'z_entry':>8} | {'train':>7} {'test':>7} | {'d.train':>7} {'d.test':>7}")
print("-" * 48)
for _ze in [0.08, 0.12, 0.18, 0.27, 0.35, 0.45, 0.60]:
    _tc = trade_cfg._replace(
        z_entry=_ze, z_exit=min(trade_cfg.z_exit, 0.85 * _ze),
        z_stop=max(trade_cfg.z_stop, 1.25 * _ze),
    )
    _r = _run(_tc)
    _mark = "  <- seeded" if abs(_ze - trade_cfg.z_entry) < 1e-9 else ""
    print(f"{_ze:8.2f} | {_r[0]:7.2f} {_r[1]:7.2f} | {_r[2]:7d} {_r[3]:7d}{_mark}")
